In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix

# Define the 1D-CNN-AE model
def build_autoencoder(input_shape):
    model = tf.keras.Sequential([
        layers.Conv1D(32, kernel_size=3, activation='relu', padding='same', input_shape=input_shape),
        layers.MaxPooling1D(pool_size=2, padding='same'),
        layers.Conv1D(16, kernel_size=3, activation='relu', padding='same'),
        layers.MaxPooling1D(pool_size=2, padding='same'),
        layers.Conv1D(8, kernel_size=3, activation='relu', padding='same'),
        layers.MaxPooling1D(pool_size=2, padding='same'),
        layers.Conv1D(4, kernel_size=3, activation='relu', padding='same'),
        layers.UpSampling1D(size=2),
        layers.Conv1D(8, kernel_size=3, activation='relu', padding='same'),
        layers.UpSampling1D(size=2),
        layers.Conv1D(16, kernel_size=3, activation='relu', padding='same'),
        layers.UpSampling1D(size=2),
        layers.Conv1D(32, kernel_size=3, activation='relu', padding='same'),
        layers.Conv1D(1, kernel_size=3, activation='linear', padding='same')
    ])
    return model


def load_data(train_path, test_path):
    train_data = pd.read_csv(train_path)
    test_data = pd.read_csv(test_path)
    
    # Drop rows with non-numeric values
    train_data = train_data.dropna().reset_index(drop=True)
    test_data = test_data.dropna().reset_index(drop=True)
    
    return train_data, test_data




In [2]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Preprocess the data
def preprocess_data(data, max_seq_length=None):
    # Perform any necessary data preprocessing steps
    # For example, scaling, filling missing values, etc.
    # You may need to adapt this depending on your data
    
    # Pad sequences to a fixed length
    data_padded = pad_sequences(data, maxlen=max_seq_length, padding='post', dtype='float32')
    return data_padded


In [3]:
def main():
    train_path = "C:\\Users\\olufe\\projects\\Journal\\dataset\\psa_journal_home_A_unsupervise_train.csv"
    test_path = "C:\\Users\\olufe\\projects\\Journal\\dataset\\psa_journal_home_A_sim_test.csv"
    
    train_data, test_data = load_data(train_path, test_path)
    
    # Preprocess data
    max_seq_length = max(train_data.shape[1], test_data.shape[1])  # Get the maximum sequence length
    X_train = preprocess_data(train_data, max_seq_length)
    X_test = preprocess_data(test_data, max_seq_length)
    
    # Initialize k-fold cross-validation
    k = 5
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    
    metrics_list = []
    
    for train_index, val_index in kf.split(X_train):
        X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
        
        # Build and train the model
        input_shape = (X_train_fold.shape[1], 1)  # Assumes data is in a 1D array
        model = build_autoencoder(input_shape)
        model = train_model(model, X_train_fold, X_val_fold)
        
        # Evaluate the model on the test data
        mse_scores = evaluate_model(model, X_test)
        
        # Compute performance metrics
        y_true = np.zeros_like(mse_scores)  # Ground truth for the test data (all 0s)
        metrics = compute_metrics(y_true, mse_scores)
        metrics_list.append(metrics)
        
    # Compute average metrics across all folds
    avg_metrics = np.mean(metrics_list, axis=0)
    print("Average performance metrics across all folds:")
    print("Accuracy:", avg_metrics[0])
    print("Precision:", avg_metrics[1])
    print("Recall:", avg_metrics[2])
    print("TNR:", avg_metrics[3])
    print("FPR:", avg_metrics[4])
    print("FNR:", avg_metrics[5])
    print("F1-score:", avg_metrics[6])
    print("AUC:", avg_metrics[7])

if __name__ == "__main__":
    main()


ValueError: could not convert string to float: 'M0'

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, roc_auc_score, f1_score
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, UpSampling1D
from tensorflow.keras.models import Model

# Load the data and convert non-numeric values to numeric using label encoding
def load_data(file_path):
    data = pd.read_csv(file_path)
    label_encoder = LabelEncoder()
    for col in data.columns:
        if data[col].dtype == 'object':
            data[col] = label_encoder.fit_transform(data[col])
    return data.values

# Define the 1D Convolutional Autoencoder model
def create_autoencoder(input_shape):
    input_layer = Input(shape=input_shape)
    encoded = Conv1D(32, 3, activation='relu', padding='same')(input_layer)
    encoded = MaxPooling1D(2, padding='same')(encoded)
    decoded = Conv1D(32, 3, activation='relu', padding='same')(encoded)
    decoded = UpSampling1D(2)(decoded)
    autoencoder = Model(input_layer, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder

# Evaluate the model using various performance metrics
def evaluate_model(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    tnr = tn / (tn + fp)
    fpr = fp / (tn + fp)
    fnr = fn / (fn + tp)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    return acc, prec, rec, tnr, fpr, fnr, f1, auc

# Main function to train and evaluate the model
def main():
    data_path = "C:/Users/olufe/projects/Journal/dataset/"
    train_file = data_path + "psa_journal_home_A_unsupervise_train.csv"
    test_file = data_path + "psa_journal_home_A_sim_test.csv"
    input_shape = (num_features, 1)

    # Load the data and labels
    train_data = load_data(train_file)
    test_data = load_data(test_file)
    num_features = train_data.shape[1] - 1  # Exclude the target column

    # Separate features and labels
    X_train, y_train = train_data[:, :-1], train_data[:, -1]
    X_test, y_test = test_data[:, :-1], test_data[:, -1]

    # Reshape data for Conv1D input
    X_train = X_train.reshape(X_train.shape[0], num_features, 1)
    X_test = X_test.reshape(X_test.shape[0], num_features, 1)

    # Create the autoencoder model
    autoencoder = create_autoencoder(input_shape)

    # Perform k-fold cross-validation
    num_folds = 5
    kf = KFold(n_splits=num_folds, shuffle=True)

    acc_list, prec_list, rec_list, tnr_list, fpr_list, fnr_list, f1_list, auc_list = [], [], [], [], [], [], [], []

    for train_index, val_index in kf.split(X_train):
        X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
        y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

        # Train the autoencoder
        autoencoder.fit(X_train_fold, X_train_fold, epochs=10, batch_size=32, validation_data=(X_val_fold, X_val_fold))

        # Use the trained autoencoder to make predictions
        X_train_pred = autoencoder.predict(X_train_fold)
        X_val_pred = autoencoder.predict(X_val_fold)

        # Calculate reconstruction error (MSE)
        train_mse = np.mean(np.square(X_train_fold - X_train_pred), axis=1)
        val_mse = np.mean(np.square(X_val_fold - X_val_pred), axis=1)

        # Set a threshold for anomaly detection (you can tune this threshold based on validation results)
        threshold = np.mean(val_mse) + np.std(val_mse)

        # Apply the threshold to label anomalies
        y_train_pred = (train_mse > threshold).astype(int)
        y_val_pred = (val_mse > threshold).astype(int)

        # Evaluate the model for this fold
        acc, prec, rec, tnr, fpr, fnr, f1, auc = evaluate_model(y_train_fold, y_train_pred)
        acc_list.append(acc)
        prec_list.append(prec)
        rec_list.append(rec)
        tnr_list.append(tnr)
        fpr_list.append(fpr)
        fnr_list.append(fnr)
        f1_list.append(f1)
        auc_list.append(auc)

    # Calculate average performance metrics over all folds
    avg_acc = np.mean(acc_list)
    avg_prec = np.mean(prec_list)
    avg_rec = np.mean(rec_list)
    avg_tnr = np.mean(tnr_list)
    avg_fpr = np.mean(fpr_list)
    avg_fnr = np.mean(fnr_list)
    avg_f1 = np.mean(f1_list)
    avg_auc = np.mean(auc_list)

    print("Average Performance Metrics over {} folds:".format(num_folds))
    print("Accuracy:", avg_acc)
    print("Precision:", avg_prec)
    print("Recall:", avg_rec)
    print("TNR (Specificity):", avg_tnr)
    print("FPR:", avg_fpr)
    print("FNR:", avg_fnr)
    print("F1-score:", avg_f1)
    print("AUC:", avg_auc)

if __name__ == "__main__":
    main()


UnboundLocalError: local variable 'num_features' referenced before assignment